# 01: ISCX VPN-nonVPN Dataset Exploration

**Thesis:** *Impact of Machine-Learning Traffic Classification Errors on Multi-Class Capacity Dimensioning in Multirate Loss Systems*

**Dataset:** ISCX VPN-nonVPN 2016 (Draper-Gil et al., 2016), Scenario B time-based features, 15-second aggregation window.

This notebook loads the raw ARFF data, filters to the 5-class subset used in the thesis,
cleans known data quality issues, and produces summary statistics and visualisations
for Chapter 4.

## 1. Setup and Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.io.arff import loadarff
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif
from pathlib import Path

%matplotlib inline
sns.set_style('whitegrid')

DATA_DIR = Path('..') / 'data' / 'Scenario B-ARFF'
PROCESSED_DIR = Path('..') / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ARFF_15S = DATA_DIR / 'TimeBasedFeatures-Dataset-15s-AllinOne.arff'

# Working palette for this notebook only (seaborn Set2); the thesis figures use the Okabe-Ito palette in src/figures/style.py
CLASS_ORDER = ['VoIP', 'Chat', 'Browsing', 'FileTransfer', 'Streaming']
CLASS_COLORS = dict(zip(CLASS_ORDER, sns.color_palette('Set2', 5)))

# Bandwidth demands (t_k) in units, from thesis specification
TK_MAP = {'VoIP': 1, 'Chat': 1, 'Browsing': 2, 'FileTransfer': 8, 'Streaming': 15}

print('Setup complete.')

In [ ]:
raw_data, meta = loadarff(ARFF_15S)
df_raw = pd.DataFrame(raw_data)

df_raw['class1'] = df_raw['class1'].str.decode('utf-8')

FEATURE_COLS = [c for c in df_raw.columns if c != 'class1']

print(f'Loaded {ARFF_15S.name}')
print(f'Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns ({len(FEATURE_COLS)} features + 1 label)')
print(f'Feature dtypes: {df_raw[FEATURE_COLS].dtypes.unique()}')
df_raw.head()

## 2. Raw Dataset Overview

In [ ]:
# 7-class distribution
raw_counts = df_raw['class1'].value_counts()
raw_pcts = df_raw['class1'].value_counts(normalize=True) * 100

dist_7class = pd.DataFrame({'count': raw_counts, 'percentage': raw_pcts.round(2)})
dist_7class.index.name = 'class'
print('Full 7-class distribution:')
print(dist_7class.to_string())
print(f'\nTotal flows: {len(df_raw)}')

In [ ]:
# Basic statistics for all features
print('Feature summary statistics:')
df_raw[FEATURE_COLS].describe().T

In [ ]:
# Check for NaN, inf, and -1 sentinel values
nan_counts = df_raw[FEATURE_COLS].isna().sum()
inf_counts = df_raw[FEATURE_COLS].apply(lambda s: np.isinf(s).sum())
neg1_counts = df_raw[FEATURE_COLS].apply(lambda s: (s == -1).sum())

quality_check = pd.DataFrame({
    'NaN': nan_counts,
    'inf': inf_counts,
    'sentinel_-1': neg1_counts
})
# Only show features with at least one issue
issues = quality_check[quality_check.sum(axis=1) > 0]
if len(issues) > 0:
    print(f'{len(issues)} features have quality issues:')
    print(issues.to_string())
else:
    print('No NaN, inf, or -1 sentinel values found.')

print(f'\nTotal -1 sentinel values across all features: {neg1_counts.sum()}')

## 3. Five-Class Filtering and Mapping

Following the FlowPic/DISTILLER convention, the thesis uses 5 of the 7 original classes.
MAIL and P2P are dropped: MAIL has too few samples for reliable classification,
and P2P is not relevant to the IPTV/OTT service scenario modelled in the thesis.

In [ ]:
# Classes to keep (original ARFF labels)
KEEP_CLASSES = ['VOIP', 'CHAT', 'BROWSING', 'FT', 'STREAMING']

# Mapping from ARFF labels to thesis convention
LABEL_MAP = {
    'VOIP': 'VoIP',
    'CHAT': 'Chat',
    'BROWSING': 'Browsing',
    'FT': 'FileTransfer',
    'STREAMING': 'Streaming'
}

# Filter
mask = df_raw['class1'].isin(KEEP_CLASSES)
df = df_raw[mask].copy()
dropped = len(df_raw) - len(df)

# Rename
df['class1'] = df['class1'].map(LABEL_MAP)

print(f'Kept {len(df)} flows, dropped {dropped} (MAIL: {raw_counts.get("MAIL", 0)}, P2P: {raw_counts.get("P2P", 0)})')
print(f'Remaining fraction: {len(df)/len(df_raw)*100:.1f}%')

In [ ]:
# 5-class distribution
class5_counts = df['class1'].value_counts().reindex(CLASS_ORDER)
class5_pcts = (class5_counts / class5_counts.sum() * 100).round(2)

dist_5class = pd.DataFrame({'count': class5_counts, 'percentage': class5_pcts})
dist_5class.index.name = 'class'
print('5-class distribution:')
print(dist_5class.to_string())
print(f'\nTotal flows: {class5_counts.sum()}')

In [ ]:
# Side-by-side comparison: 7-class vs 5-class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 7-class
raw_counts.sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Original 7 classes')
axes[0].set_xlabel('Number of flows')

# 5-class
class5_counts.plot.barh(ax=axes[1], color=[CLASS_COLORS[c] for c in CLASS_ORDER])
axes[1].set_title('Filtered 5 classes (thesis subset)')
axes[1].set_xlabel('Number of flows')

plt.tight_layout()
plt.show()

## 4. Data Quality Analysis

Known issues in the ISCX VPN-nonVPN dataset:

1. **Sentinel values (`-1`):** Ten features use `-1` as a sentinel for unidirectional flows
   or inactive periods. This affects IAT features (`total_fiat`, `min_fiat`, `total_biat`,
   `min_biat`, `min_flowiat`, `max_flowiat`) and active/idle features (`min_active`,
   `max_active`, `min_idle`, `max_idle`). These are replaced with `0`
   because such flows contribute zero time in the missing direction/period.

2. **Extreme outliers:** Short-duration flows can produce very large `flowBytesPerSecond`
   and `flowPktsPerSecond` values. Every feature is clipped at its own 99.9th percentile.

In [ ]:
# Count -1 sentinel values per feature in the 5-class subset
neg1_per_feat = df[FEATURE_COLS].apply(lambda s: (s == -1).sum())
affected = neg1_per_feat[neg1_per_feat > 0]

print(f'Features with -1 sentinel values ({len(affected)}/{len(FEATURE_COLS)}):')
print(affected.to_string())
print(f'\nTotal -1 values in 5-class subset: {affected.sum()}')

In [ ]:
# Replace -1 sentinels with 0
sentinel_cols = affected.index.tolist()
for col in sentinel_cols:
    df[col] = df[col].replace(-1, 0)

# Verify
remaining_neg1 = df[FEATURE_COLS].apply(lambda s: (s == -1).sum()).sum()
print(f'Remaining -1 values after replacement: {remaining_neg1}')

In [ ]:
# Check for infinities and very large values before clipping
inf_count = df[FEATURE_COLS].apply(lambda s: np.isinf(s).sum()).sum()
print(f'Infinite values: {inf_count}')

# Distribution of flowBytesPerSecond and flowPktsPerSecond before clipping
rate_features = ['flowBytesPerSecond', 'flowPktsPerSecond']
print('\nBefore clipping:')
print(df[rate_features].describe().to_string())

In [ ]:
# Clip all numeric features at the 99.9th percentile
clip_thresholds = {}
for col in FEATURE_COLS:
    p999 = df[col].quantile(0.999)
    n_clipped = (df[col] > p999).sum()
    if n_clipped > 0:
        clip_thresholds[col] = {'threshold': p999, 'n_clipped': n_clipped}
    df[col] = df[col].clip(upper=p999)

print(f'Features clipped at 99.9th percentile ({len(clip_thresholds)} affected):')
clip_df = pd.DataFrame(clip_thresholds).T
print(clip_df.to_string())

print('\nAfter clipping:')
print(df[rate_features].describe().to_string())

In [ ]:
# Before/after comparison for rate features
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i, feat in enumerate(rate_features):
    # Before (from raw filtered data)
    raw_vals = df_raw.loc[mask, feat]
    axes[i, 0].hist(raw_vals, bins=80, color='salmon', edgecolor='white', alpha=0.8)
    axes[i, 0].set_title(f'{feat} (before clipping)')
    axes[i, 0].set_ylabel('Frequency')
    
    # After
    axes[i, 1].hist(df[feat], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i, 0].set_yscale('symlog', linthresh=1)
    axes[i, 1].set_yscale('symlog', linthresh=1)
    axes[i, 1].set_title(f'{feat} (after clipping at 99.9th pct)')

plt.tight_layout()
plt.show()

In [ ]:
# Final quality check
nan_final = df[FEATURE_COLS].isna().sum().sum()
inf_final = df[FEATURE_COLS].apply(lambda s: np.isinf(s).sum()).sum()
neg1_final = df[FEATURE_COLS].apply(lambda s: (s == -1).sum()).sum()

print('Final data quality check:')
print(f'  NaN values:  {nan_final}')
print(f'  Inf values:  {inf_final}')
print(f'  -1 sentinels: {neg1_final}')
print(f'  Shape: {df.shape}')

## 5. Feature Distributions

In [ ]:
# 4x6 grid, 24 subplots, the last one hidden
fig, axes = plt.subplots(4, 6, figsize=(20, 14))
axes_flat = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    axes_flat[i].hist(df[col], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    # log-y to compress the long tail; many features are heavy-tailed (durations, byte counts)
    axes_flat[i].set_yscale('symlog', linthresh=1)
    axes_flat[i].set_title(col, fontsize=9)
    axes_flat[i].tick_params(labelsize=7)

axes_flat[-1].set_visible(False)

fig.suptitle('Feature distributions (5-class subset, after cleaning)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots by class for selected features
box_features = ['flowBytesPerSecond', 'mean_flowiat', 'std_flowiat', 'duration', 'flowPktsPerSecond']

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
for i, feat in enumerate(box_features):
    sns.boxplot(
        data=df, x='class1', y=feat, order=CLASS_ORDER,
        palette=CLASS_COLORS, ax=axes[i], fliersize=2
    )
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30, labelsize=8)

fig.suptitle('Per-class box plots for selected features', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (23x23)
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
    square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 7}
)
ax.set_title('Feature correlation matrix (23 features)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Per-Class Feature Analysis

In [ ]:
# Per-class mean and standard deviation for all features
class_means = df.groupby('class1')[FEATURE_COLS].mean().reindex(CLASS_ORDER)
class_stds = df.groupby('class1')[FEATURE_COLS].std().reindex(CLASS_ORDER)

print('Per-class feature means:')
class_means.T

In [ ]:
# ANOVA F-statistic to identify most discriminative features
X = df[FEATURE_COLS].values
y = df['class1'].values

f_scores, p_values = f_classif(X, y)

anova_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'F_score': f_scores,
    'p_value': p_values
}).sort_values('F_score', ascending=False)

print('ANOVA F-statistics (all features, sorted by discriminative power):')
print(anova_df.to_string(index=False))

In [ ]:
# Violin plots for the top-3 most discriminative features
top3 = anova_df.head(3)['feature'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(top3):
    sns.violinplot(
        data=df, x='class1', y=feat, order=CLASS_ORDER,
        palette=CLASS_COLORS, ax=axes[i], inner='box', cut=0
    )
    f_val = anova_df.loc[anova_df['feature'] == feat, 'F_score'].iloc[0]
    axes[i].set_title(f'{feat}\n(F = {f_val:.1f})', fontsize=10)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30, labelsize=8)

fig.suptitle('Violin plots: top-3 discriminative features (by ANOVA F-score)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 7. Summary Statistics

Tables and mappings used in Chapter 4.

In [ ]:
# Summary table for Ch.4
summary = pd.DataFrame({
    'count': class5_counts,
    'percentage': class5_pcts,
    'mean_flowBytesPerSecond': df.groupby('class1')['flowBytesPerSecond'].mean().reindex(CLASS_ORDER).round(1),
    'mean_duration': df.groupby('class1')['duration'].mean().reindex(CLASS_ORDER).round(1)
})
summary.index.name = 'class'

print(f'Total flows (5-class): {class5_counts.sum()}')
print()
print(summary.to_string())

In [ ]:
# 5-class mapping table with bandwidth demands (t_k)
mapping_table = pd.DataFrame({
    'ARFF_label': ['VOIP', 'CHAT', 'BROWSING', 'FT', 'STREAMING'],
    'thesis_label': CLASS_ORDER,
    't_k (bandwidth units)': [TK_MAP[c] for c in CLASS_ORDER]
})
print('Class mapping with bandwidth demands (t_k):')
print(mapping_table.to_string(index=False))

In [ ]:
# Save cleaned DataFrame
output_path = PROCESSED_DIR / 'iscx_5class_15s_clean.csv'
df.to_csv(output_path, index=False)
print(f'Saved cleaned data to {output_path}')
print(f'Shape: {df.shape}')

## 8. Comparison Across Time Windows (Optional)

The ISCX VPN-nonVPN Scenario B data is provided at four aggregation windows:
15 s, 30 s, 60 s, and 120 s. The 15 s window was chosen for the thesis because
it yields the most flows and is the standard interval used in FlowPic and DISTILLER.

In [ ]:
# Note: some ARFF files (e.g. 60s) contain malformed rows that scipy cannot parse. These are skipped with a warning.
windows = ['15s', '30s', '60s', '120s']
window_stats = []

for w in windows:
    path = DATA_DIR / f'TimeBasedFeatures-Dataset-{w}-AllinOne.arff'
    try:
        data_w, _ = loadarff(path)
    except ValueError as e:
        print(f'WARNING: skipping {w} window (ARFF parse error: {e})')
        continue
    df_w = pd.DataFrame(data_w)
    df_w['class1'] = df_w['class1'].str.decode('utf-8')
    
    # Filter to 5 classes
    df_w = df_w[df_w['class1'].isin(KEEP_CLASSES)].copy()
    df_w['class1'] = df_w['class1'].map(LABEL_MAP)
    
    per_class = df_w['class1'].value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)
    stats = {
        'window': w,
        'total_flows': len(df_w),
    }
    for c in CLASS_ORDER:
        stats[c] = per_class.get(c, 0)
    window_stats.append(stats)

tw_df = pd.DataFrame(window_stats).set_index('window')
print('Flow counts by time window and class (5-class subset):')
print(tw_df.to_string())

In [ ]:
# Visual comparison of total flows by window
fig, ax = plt.subplots(figsize=(7, 4))
tw_df['total_flows'].plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_ylabel('Number of flows')
ax.set_xlabel('Aggregation window')
ax.set_title('Total 5-class flows by time window')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(tw_df['total_flows']):
    ax.text(i, v + 100, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Next step:** `02_classifiers.ipynb` trains the XGBoost and MLP classifiers on the
cleaned 5-class dataset produced here and writes the confusion-matrix archive that
the analytical chapters read.